In [2]:
import pandas as pd
import json
import pyarrow as pa
import pyarrow.parquet as pq
import requests



In [7]:
url = ("https://raw.githubusercontent.com/mhollingshead/billboard-hot-100/main/recent.json")
query_parameters = {"downloadformat": "csv"}
response = requests.get(url, params=query_parameters)


In [9]:
response.content

b'{"date":"2025-12-13","data":[{"song":"All I Want For Christmas Is You","artist":"Mariah Carey","this_week":1,"last_week":5,"peak_position":1,"weeks_on_chart":76},{"song":"Last Christmas","artist":"Wham!","this_week":2,"last_week":6,"peak_position":2,"weeks_on_chart":50},{"song":"Rockin\' Around The Christmas Tree","artist":"Brenda Lee","this_week":3,"last_week":7,"peak_position":1,"weeks_on_chart":68},{"song":"Jingle Bell Rock","artist":"Bobby Helms","this_week":4,"last_week":8,"peak_position":3,"weeks_on_chart":65},{"song":"Golden","artist":"HUNTR/X: EJAE, Audrey Nuna & REI AMI","this_week":5,"last_week":2,"peak_position":1,"weeks_on_chart":24},{"song":"The Fate Of Ophelia","artist":"Taylor Swift","this_week":6,"last_week":1,"peak_position":1,"weeks_on_chart":9},{"song":"Ordinary","artist":"Alex Warren","this_week":7,"last_week":3,"peak_position":1,"weeks_on_chart":43},{"song":"Santa Tell Me","artist":"Ariana Grande","this_week":8,"last_week":13,"peak_position":5,"weeks_on_chart":33

In [13]:
data = json.loads(response.content)
s = json.dumps(data, indent=4, sort_keys=True)
file3 = pd.read_json(s)

/tmp/ipykernel_835/1196471522.py:3: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  file3 = pd.read_json(s)


In [15]:
display(pd.json_normalize(file3['data']))

,artist,last_week,peak_position,song,this_week,weeks_on_chart
0,Mariah Carey,5.0,1,All I Want For Christmas Is You,1,76
1,Wham!,6.0,2,Last Christmas,2,50
2,Brenda Lee,7.0,1,Rockin' Around The Christmas Tree,3,68
3,Bobby Helms,8.0,3,Jingle Bell Rock,4,65
4,"HUNTR/X: EJAE, Audrey Nuna & REI AMI",2.0,1,Golden,5,24
...,...,...,...,...,...,...
95,Cynthia Erivo & Ariana Grande,43.0,43,For Good,96,2
96,Cynthia Erivo,56.0,56,No Good Deed,97,2
97,Olivia Dean,95.0,95,Let Alone The One You Love,98,3
98,Taylor Swift Featuring Sabrina Carpenter,80.0,8,The Life Of A Showgirl,99,9


In [19]:
file_merged = pd.concat([file3['date'], pd.json_normalize(file3['data'])], axis=1)

In [20]:
file_merged = file_merged.iloc[:, [0,4,1,2,3,,5]]
['','','','','','','']
file_merged

,date,artist,last_week,peak_position,song,this_week,weeks_on_chart
0,2025-12-13,Mariah Carey,5.0,1,All I Want For Christmas Is You,1,76
1,2025-12-13,Wham!,6.0,2,Last Christmas,2,50
2,2025-12-13,Brenda Lee,7.0,1,Rockin' Around The Christmas Tree,3,68
3,2025-12-13,Bobby Helms,8.0,3,Jingle Bell Rock,4,65
4,2025-12-13,"HUNTR/X: EJAE, Audrey Nuna & REI AMI",2.0,1,Golden,5,24
...,...,...,...,...,...,...,...
95,2025-12-13,Cynthia Erivo & Ariana Grande,43.0,43,For Good,96,2
96,2025-12-13,Cynthia Erivo,56.0,56,No Good Deed,97,2
97,2025-12-13,Olivia Dean,95.0,95,Let Alone The One You Love,98,3
98,2025-12-13,Taylor Swift Featuring Sabrina Carpenter,80.0,8,The Life Of A Showgirl,99,9


In [21]:
billboardOAT = pd.read_parquet('raw/billboard/billboard_100_OAT.parquet', engine='pyarrow')

In [22]:
billboardOAT

,date,song,artist,this_week,last_week,peak_position,weeks_on_chart
0,1958-08-04,Poor Little Fool,Ricky Nelson,1.0,NaN,1.0,1.0
1,1958-08-04,Patricia,Perez Prado And His Orchestra,2.0,NaN,2.0,1.0
2,1958-08-04,Splish Splash,Bobby Darin,3.0,NaN,3.0,1.0
3,1958-08-04,Hard Headed Woman,Elvis Presley With The Jordanaires,4.0,NaN,4.0,1.0
4,1958-08-04,When,Kalin Twins,5.0,NaN,5.0,1.0
...,...,...,...,...,...,...,...
351295,2025-11-29,Baller,"Summer Walker, Glorilla, Sexyy Red & Monaleo",96.0,NaN,96.0,1.0
351296,2025-11-29,Let Alone The One You Love,Olivia Dean,97.0,NaN,97.0,1.0
351297,2025-11-29,Losin' Streak,Blake Roman,98.0,NaN,98.0,1.0
351298,2025-11-29,Favorite Country Song,HARDY,99.0,90.0,90.0,4.0


In [26]:
oat = pd.concat([billboardOAT, file_merged],ignore_index=True)
oat

,date,song,artist,this_week,last_week,peak_position,weeks_on_chart
0,1958-08-04,Poor Little Fool,Ricky Nelson,1.0,NaN,1.0,1.0
1,1958-08-04,Patricia,Perez Prado And His Orchestra,2.0,NaN,2.0,1.0
2,1958-08-04,Splish Splash,Bobby Darin,3.0,NaN,3.0,1.0
3,1958-08-04,Hard Headed Woman,Elvis Presley With The Jordanaires,4.0,NaN,4.0,1.0
4,1958-08-04,When,Kalin Twins,5.0,NaN,5.0,1.0
...,...,...,...,...,...,...,...
351395,2025-12-13 00:00:00,For Good,Cynthia Erivo & Ariana Grande,96.0,43.0,43.0,2.0
351396,2025-12-13 00:00:00,No Good Deed,Cynthia Erivo,97.0,56.0,56.0,2.0
351397,2025-12-13 00:00:00,Let Alone The One You Love,Olivia Dean,98.0,95.0,95.0,3.0
351398,2025-12-13 00:00:00,The Life Of A Showgirl,Taylor Swift Featuring Sabrina Carpenter,99.0,80.0,8.0,9.0


In [27]:
oat['date'] = pd.to_datetime(oat['date']).dt.date
oat

,date,song,artist,this_week,last_week,peak_position,weeks_on_chart
0,1958-08-04,Poor Little Fool,Ricky Nelson,1.0,NaN,1.0,1.0
1,1958-08-04,Patricia,Perez Prado And His Orchestra,2.0,NaN,2.0,1.0
2,1958-08-04,Splish Splash,Bobby Darin,3.0,NaN,3.0,1.0
3,1958-08-04,Hard Headed Woman,Elvis Presley With The Jordanaires,4.0,NaN,4.0,1.0
4,1958-08-04,When,Kalin Twins,5.0,NaN,5.0,1.0
...,...,...,...,...,...,...,...
351395,2025-12-13,For Good,Cynthia Erivo & Ariana Grande,96.0,43.0,43.0,2.0
351396,2025-12-13,No Good Deed,Cynthia Erivo,97.0,56.0,56.0,2.0
351397,2025-12-13,Let Alone The One You Love,Olivia Dean,98.0,95.0,95.0,3.0
351398,2025-12-13,The Life Of A Showgirl,Taylor Swift Featuring Sabrina Carpenter,99.0,80.0,8.0,9.0


# Partie Analyse et création dossier Spotify

In [6]:
file = pd.read_json("raw/spotify/artist/top_tracks_Anirudh Ravichander_2025-12-04.json")
file2 = pd.json_normalize(file.tracks)
display(file2)


,artists,disc_number,duration_ms,explicit,href,id,is_local,is_playable,name,popularity,...,album.images,album.is_playable,album.name,album.release_date,album.release_date_precision,album.total_tracks,album.type,album.uri,external_ids.isrc,external_urls.spotify
0,[{'external_urls': {'spotify': 'https://open.s...,1,217627,False,https://api.spotify.com/v1/tracks/7MrdHOL2aoWf...,7MrdHOL2aoWfT16CncgNei,False,True,Monica,68,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Coolie (Original Motion Picture Soundtrack),2025-08-02,day,8,album,spotify:album:3roEiD2gUP1LHPHwUPbXs1,QMDA62550185,https://open.spotify.com/track/7MrdHOL2aoWfT16...
1,[{'external_urls': {'spotify': 'https://open.s...,1,197104,False,https://api.spotify.com/v1/tracks/28keFvZn4UVR...,28keFvZn4UVRi4I8Rj4GjY,False,True,"Thalapathy Kacheri (From ""Jana Nayagan"")",76,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,"Thalapathy Kacheri (From ""Jana Nayagan"")",2025-11-08,day,1,album,spotify:album:0TthxobhZuRUVweIhTUwuc,INS182504154,https://open.spotify.com/track/28keFvZn4UVRi4I...
2,[{'external_urls': {'spotify': 'https://open.s...,1,206627,False,https://api.spotify.com/v1/tracks/3JC5Xx48KqzY...,3JC5Xx48KqzYYlbTr6weCv,False,True,Powerhouse,68,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Coolie (Original Motion Picture Soundtrack),2025-08-02,day,8,album,spotify:album:3roEiD2gUP1LHPHwUPbXs1,QMDA62587043,https://open.spotify.com/track/3JC5Xx48KqzYYlb...
3,[{'external_urls': {'spotify': 'https://open.s...,1,200373,False,https://api.spotify.com/v1/tracks/3xMHXmedL5Rv...,3xMHXmedL5Rvfxmiar9Ryv,False,True,Chaleya,69,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Jawan,2023-09-05,day,7,album,spotify:album:3WLJmMZUeDOuERFAk1Mxs6,INS182302348,https://open.spotify.com/track/3xMHXmedL5Rvfxm...
4,[{'external_urls': {'spotify': 'https://open.s...,1,222063,False,https://api.spotify.com/v1/tracks/1bxzr3JK05fM...,1bxzr3JK05fMTcweGAZUHp,False,True,"Chuttamalle (From ""Devara Part 1"")",74,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,"Chuttamalle (From ""Devara Part 1"")",2024-08-05,day,1,album,spotify:album:77bk7awXNSGEQkZTcTp2Gj,INS182402313,https://open.spotify.com/track/1bxzr3JK05fMTcw...
5,[{'external_urls': {'spotify': 'https://open.s...,1,188505,False,https://api.spotify.com/v1/tracks/7vKddq80IImH...,7vKddq80IImHdiz83ZU2zF,False,True,Monica,65,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Coolie (Original Motion Picture Soundtrack),2025-08-12,day,8,album,spotify:album:22mwh2WQgpVT4j5CaNGWYl,QMDA62550221,https://open.spotify.com/track/7vKddq80IImHdiz...
6,[{'external_urls': {'spotify': 'https://open.s...,1,256945,False,https://api.spotify.com/v1/tracks/4KUQFOGTaHrg...,4KUQFOGTaHrggl3hfMoaAe,False,True,"Kannamma - From ""Ispade Rajavum Idhaya Raniyum""",67,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,"Kannamma (From ""Ispade Rajavum Idhaya Raniyum"")",2019-01-11,day,1,album,spotify:album:7bgFFKluLbu4PRIRzdink9,INT201901935,https://open.spotify.com/track/4KUQFOGTaHrggl3...
7,[{'external_urls': {'spotify': 'https://open.s...,1,235629,False,https://api.spotify.com/v1/tracks/3CA9CjRf7IU2...,3CA9CjRf7IU2Ukz3GuYInP,False,True,Manasilaayo,68,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Vettaiyan (Original Motion Picture Soundtrack),2024-10-14,day,9,album,spotify:album:6Xw7XAQQwpsy9KfUnJTrHz,INS172409897,https://open.spotify.com/track/3CA9CjRf7IU2Ukz...
8,[{'external_urls': {'spotify': 'https://open.s...,1,240845,False,https://api.spotify.com/v1/tracks/2unhBSPWS4Cp...,2unhBSPWS4Cpn5EfIKbAxG,False,True,Thangapoovey,56,...,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Madharaasi (Original Motion Picture Soundtrack),2025-08-27,day,9,album,spotify:album:7HUzJeQvFdJDWueJi6pofV,INT132500505,https://open.spotify.com/track/2unhBSPWS4Cpn5E...
9,[{'external_urls': {'spotify': 'https://open.s...,1,207219,False,https://api.spotify.com/v1/tracks/07dj2zpMixBX...,07dj2zpMixBXxQuwp

In [3]:
file2.iloc[:,10:19]

,preview_url,track_number,type,uri,album.album_type,album.artists,album.external_urls.spotify,album.href,album.id
0,None,1,track,spotify:track:2BwO5K8Q7EPAJSGze3AAh9,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/1aqg30bNvLSWgSh...,https://api.spotify.com/v1/albums/1aqg30bNvLSW...,1aqg30bNvLSWgShZgX4oop
1,None,2,track,spotify:track:42VUCXerQ5qTr4Qp6PhKo4,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/1aqg30bNvLSWgSh...,https://api.spotify.com/v1/albums/1aqg30bNvLSW...,1aqg30bNvLSWgShZgX4oop
2,None,12,track,spotify:track:5eXgqtg3T8Av0m1FUaGHex,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/4a6NzYL1YHRUgx9...,https://api.spotify.com/v1/albums/4a6NzYL1YHRU...,4a6NzYL1YHRUgx9e3YZI6I
3,None,7,track,spotify:track:2HRqTpkrJO5ggZyyK6NPWz,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/3iPSVi54hsacKKl...,https://api.spotify.com/v1/albums/3iPSVi54hsac...,3iPSVi54hsacKKl1xIR2eH
4,None,8,track,spotify:track:0je57Uq5eTk1wrPzn9sWbl,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/1aqg30bNvLSWgSh...,https://api.spotify.com/v1/albums/1aqg30bNvLSW...,1aqg30bNvLSWgShZgX4oop
5,None,1,track,spotify:track:5G2f63n7IPVPPjfNIGih7Q,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/3iPSVi54hsacKKl...,https://api.spotify.com/v1/albums/3iPSVi54hsac...,3iPSVi54hsacKKl1xIR2eH
6,None,6,track,spotify:track:4SRShYMtFIGgnOU7iBicMH,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/1aqg30bNvLSWgSh...,https://api.spotify.com/v1/albums/1aqg30bNvLSW...,1aqg30bNvLSWgShZgX4oop
7,None,2,track,spotify:track:2tHwzyyOLoWSFqYNjeVMzj,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/3iPSVi54hsacKKl...,https://api.spotify.com/v1/albums/3iPSVi54hsac...,3iPSVi54hsacKKl1xIR2eH
8,None,11,track,spotify:track:25jgQBxuUkGDdCG1WGKKN9,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/1aqg30bNvLSWgSh...,https://api.spotify.com/v1/albums/1aqg30bNvLSW...,1aqg30bNvLSWgShZgX4oop
9,None,9,track,spotify:track:6dgUya35uo964z7GZXM07g,album,[{'external_urls': {'spotify': 'https://open.s...,https://open.spotify.com/album/5kDmlA2g9Y1YCbN...,https://api.spotify.com/v1/albums/5kDmlA2g9Y1Y...,5kDmlA2g9Y1YCbNo2Ufxlz


In [4]:
file2.iloc[:,19:29]

,album.images,album.is_playable,album.name,album.release_date,album.release_date_precision,album.total_tracks,album.type,album.uri,external_ids.isrc,external_urls.spotify
0,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Man’s Best Friend,2025-08-29,day,12,album,spotify:album:1aqg30bNvLSWgShZgX4oop,USUM72504354,https://open.spotify.com/track/2BwO5K8Q7EPAJSG...
1,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Man’s Best Friend,2025-08-29,day,12,album,spotify:album:1aqg30bNvLSWgShZgX4oop,USUM72504355,https://open.spotify.com/track/42VUCXerQ5qTr4Q...
2,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,The Life of a Showgirl,2025-10-03,day,12,album,spotify:album:4a6NzYL1YHRUgx9e3YZI6I,USUG12506447,https://open.spotify.com/track/5eXgqtg3T8Av0m1...
3,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Short n' Sweet,2024-08-23,day,12,album,spotify:album:3iPSVi54hsacKKl1xIR2eH,USUM72403305,https://open.spotify.com/track/2HRqTpkrJO5ggZy...
4,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Man’s Best Friend,2025-08-29,day,12,album,spotify:album:1aqg30bNvLSWgShZgX4oop,USUM72504446,https://open.spotify.com/track/0je57Uq5eTk1wrP...
5,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Short n' Sweet,2024-08-23,day,12,album,spotify:album:3iPSVi54hsacKKl1xIR2eH,USUM72404100,https://open.spotify.com/track/5G2f63n7IPVPPjf...
6,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Man’s Best Friend,2025-08-29,day,12,album,spotify:album:1aqg30bNvLSWgShZgX4oop,USUM72504443,https://open.spotify.com/track/4SRShYMtFIGgnOU...
7,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Short n' Sweet,2024-08-23,day,12,album,spotify:album:3iPSVi54hsacKKl1xIR2eH,USUM72404101,https://open.spotify.com/track/2tHwzyyOLoWSFqY...
8,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,Man’s Best Friend,2025-08-29,day,12,album,spotify:album:1aqg30bNvLSWgShZgX4oop,USUM72504449,https://open.spotify.com/track/25jgQBxuUkGDdCG...
9,"[{'height': 640, 'url': 'https://i.scdn.co/ima...",True,emails i can't send,2022-07-15,day,13,album,spotify:album:5kDmlA2g9Y1YCbNo2Ufxlz,USUM72210708,https://open.spotify.com/track/6dgUya35uo964z7...


In [5]:
pd.json_normalize(file2.artists[0])['external_urls.spotify'].values


array(['https://open.spotify.com/artist/74KM79TiuVKeVCqs8QtB0B'],
      dtype=object)

In [6]:
pd.json_normalize(file2.artists[1])

,href,id,name,type,uri,external_urls.spotify
0,https://api.spotify.com/v1/artists/74KM79TiuVK...,74KM79TiuVKeVCqs8QtB0B,Sabrina Carpenter,artist,spotify:artist:74KM79TiuVKeVCqs8QtB0B,https://open.spotify.com/artist/74KM79TiuVKeVC...


In [7]:
file2["href"].values

array(['https://api.spotify.com/v1/tracks/2BwO5K8Q7EPAJSGze3AAh9',
       'https://api.spotify.com/v1/tracks/42VUCXerQ5qTr4Qp6PhKo4',
       'https://api.spotify.com/v1/tracks/5eXgqtg3T8Av0m1FUaGHex',
       'https://api.spotify.com/v1/tracks/2HRqTpkrJO5ggZyyK6NPWz',
       'https://api.spotify.com/v1/tracks/0je57Uq5eTk1wrPzn9sWbl',
       'https://api.spotify.com/v1/tracks/5G2f63n7IPVPPjfNIGih7Q',
       'https://api.spotify.com/v1/tracks/4SRShYMtFIGgnOU7iBicMH',
       'https://api.spotify.com/v1/tracks/2tHwzyyOLoWSFqYNjeVMzj',
       'https://api.spotify.com/v1/tracks/25jgQBxuUkGDdCG1WGKKN9',
       'https://api.spotify.com/v1/tracks/6dgUya35uo964z7GZXM07g'],
      dtype=object)

In [8]:
pd.json_normalize(file2.artists[1])

,href,id,name,type,uri,external_urls.spotify
0,https://api.spotify.com/v1/artists/74KM79TiuVK...,74KM79TiuVKeVCqs8QtB0B,Sabrina Carpenter,artist,spotify:artist:74KM79TiuVKeVCqs8QtB0B,https://open.spotify.com/artist/74KM79TiuVKeVC...


In [9]:
pd.json_normalize(file2['album.images'])

,0,1,2
0,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
1,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
2,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
3,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
4,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
5,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
6,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
7,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
8,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."
9,"{'height': 640, 'url': 'https://i.scdn.co/imag...","{'height': 300, 'url': 'https://i.scdn.co/imag...","{'height': 64, 'url': 'https://i.scdn.co/image..."


In [10]:
pd.json_normalize(pd.json_normalize(file2['album.images'])[0]).values


array([[640,
        'https://i.scdn.co/image/ab67616d0000b273b1863bf95557ea7f357c4947',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273b1863bf95557ea7f357c4947',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273d7812467811a7da6e6a44902',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273fd8d7a8d96871e791cb1f626',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273b1863bf95557ea7f357c4947',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273fd8d7a8d96871e791cb1f626',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273b1863bf95557ea7f357c4947',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273fd8d7a8d96871e791cb1f626',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273b1863bf95557ea7f357c4947',
        640],
       [640,
        'https://i.scdn.co/image/ab67616d0000b273700f7bf79c9

In [11]:
pd.json_normalize(file2['album.images'][1])

,height,url,width
0,640,https://i.scdn.co/image/ab67616d0000b273b1863b...,640
1,300,https://i.scdn.co/image/ab67616d00001e02b1863b...,300
2,64,https://i.scdn.co/image/ab67616d00004851b1863b...,64


# Partie Analyse et création dossier Billboard

In [3]:
file_new = pd.read_json("all.json")

In [4]:
display(file_new)

,date,data
0,1958-08-04,"[{'song': 'Poor Little Fool', 'artist': 'Ricky..."
1,1958-08-11,"[{'song': 'Poor Little Fool', 'artist': 'Ricky..."
2,1958-08-18,"[{'song': 'Nel Blu Dipinto Di Blu (Volare)', '..."
3,1958-08-25,"[{'song': 'Little Star', 'artist': 'The Elegan..."
4,1958-09-01,"[{'song': 'Nel Blu Dipinto Di Blu (Volare)', '..."
...,...,...
3510,2025-11-15,"[{'song': 'The Fate Of Ophelia', 'artist': 'Ta..."
3511,2025-11-22,"[{'song': 'The Fate Of Ophelia', 'artist': 'Ta..."
3512,2025-11-29,"[{'song': 'The Fate Of Ophelia', 'artist': 'Ta..."
3513,2025-12-06,"[{'song': 'The Fate Of Ophelia', 'artist': 'Ta..."


In [5]:
pd.json_normalize(file_new.data)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,"{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Splish Splash', 'artist': 'Bobby Dar...","{'song': 'Hard Headed Woman', 'artist': 'Elvis...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...","{'song': 'Rebel-'rouser', 'artist': 'Duane Edd...","{'song': 'Yakety Yak', 'artist': 'The Coasters...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Willie And The Hand Jive', 'artist':...","{'song': 'Fever', 'artist': 'Peggy Lee', 'this...",...,{'song': 'The Purple People Eater Meets The Wi...,"{'song': 'Bird Dog', 'artist': 'The Everly Bro...","{'song': 'Are You Really Mine', 'artist': 'Jim...",{'song': 'She Was Only Seventeen (He Was One Y...,"{'song': 'Little Mary', 'artist': 'Fats Domino...","{'song': 'Over And Over', 'artist': 'Thurston ...","{'song': 'I Believe In You', 'artist': 'Robert...","{'song': 'Little Serenade', 'artist': 'The Ame...",{'song': 'I'll Get By (As Long As I Have You)'...,"{'song': 'Judy', 'artist': 'Frankie Vaughan', ..."
1,"{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Nel Blu Dipinto Di Blu (Volare)', 'a...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Splish Splash', 'artist': 'Bobby Dar...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Hard Headed Woman', 'artist': 'Elvis...","{'song': 'Rebel-'rouser', 'artist': 'Duane Edd...","{'song': 'Just A Dream', 'artist': 'Jimmy Clan...","{'song': 'Willie And The Hand Jive', 'artist':...",...,"{'song': 'All I Have To Do Is Dream', 'artist'...","{'song': 'La Paloma', 'artist': 'Billy Vaughn ...","{'song': 'I Believe In You', 'artist': 'Robert...","{'song': 'Midnighter', 'artist': 'The Champs',...","{'song': 'Chariot Rock', 'artist': 'The Champs...","{'song': 'Down In Virginia', 'artist': 'Jimmy ...","{'song': 'Sunday Barbecue', 'artist': 'Tenness...","{'song': 'Gotta Have Rain', 'artist': 'Eydie G...","{'song': 'Nothing In The World', 'artist': 'Na...","{'song': 'Baubles, Bangles And Beads', 'artist..."
2,"{'song': 'Nel Blu Dipinto Di Blu (Volare)', 'a...","{'song': 'Little Star', 'artist': 'The Elegant...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Just A Dream', 'artist': 'Jimmy Clan...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...","{'song': 'Rebel-'rouser', 'artist': 'Duane Edd...","{'song': 'Fever', 'artist': 'Peggy Lee', 'this...","{'song': 'Splish Splash', 'artist': 'Bobby Dar...",...,"{'song': 'Down The Aisle Of Love', 'artist': '...","{'song': 'Return To Me', 'artist': 'Dean Marti...","{'song': 'Down In Virginia', 'artist': 'Jimmy ...","{'song': 'Fire Of Love', 'artist': 'Jody Reyno...","{'song': 'Put A Ring On My Finger', 'artist': ...","{'song': 'It's All In The Game', 'artist': 'To...","{'song': 'Ma Ma Ma Marie', 'artist': 'The Gayl...","{'song': 'Where The Blue Of The Night', 'artis...","{'song': 'Who Are They To Say', 'artist': 'The...","{'song': 'Going To Chicago Blues', 'artist': '..."
3,"{'song': 'Little Star', 'artist': 'The Elegant...","{'song': 'Nel Blu Dipinto Di Blu (Volare)', 'a...","{'song': 'Bird Dog', 'artist': 'The Everly Bro...","{'song': 'Just A Dream', 'artist': 'Jimmy Clan...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Born Too Late', 'artist': 'Poni-Tail...","{'song': 'Fever', 'artist': 'Peggy Lee', 'this...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...",...,"{'song': 'Ramrod', 'artist': 'Duane Eddy His T...","{'song': 'Just Like In The Movies', 'artist': ...","{'song': 'The Purple People Eater', 'artist': ...","{'song': 'You Cheated', 'artist': 'The Shields...","{'song': 'No One Knows', 'artist': 'Dion & The...","{'song': 'Return To Me', '

In [6]:
display(pd.json_normalize(file_new['data']))

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,"{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Splish Splash', 'artist': 'Bobby Dar...","{'song': 'Hard Headed Woman', 'artist': 'Elvis...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...","{'song': 'Rebel-'rouser', 'artist': 'Duane Edd...","{'song': 'Yakety Yak', 'artist': 'The Coasters...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Willie And The Hand Jive', 'artist':...","{'song': 'Fever', 'artist': 'Peggy Lee', 'this...",...,{'song': 'The Purple People Eater Meets The Wi...,"{'song': 'Bird Dog', 'artist': 'The Everly Bro...","{'song': 'Are You Really Mine', 'artist': 'Jim...",{'song': 'She Was Only Seventeen (He Was One Y...,"{'song': 'Little Mary', 'artist': 'Fats Domino...","{'song': 'Over And Over', 'artist': 'Thurston ...","{'song': 'I Believe In You', 'artist': 'Robert...","{'song': 'Little Serenade', 'artist': 'The Ame...",{'song': 'I'll Get By (As Long As I Have You)'...,"{'song': 'Judy', 'artist': 'Frankie Vaughan', ..."
1,"{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Nel Blu Dipinto Di Blu (Volare)', 'a...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Splish Splash', 'artist': 'Bobby Dar...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Hard Headed Woman', 'artist': 'Elvis...","{'song': 'Rebel-'rouser', 'artist': 'Duane Edd...","{'song': 'Just A Dream', 'artist': 'Jimmy Clan...","{'song': 'Willie And The Hand Jive', 'artist':...",...,"{'song': 'All I Have To Do Is Dream', 'artist'...","{'song': 'La Paloma', 'artist': 'Billy Vaughn ...","{'song': 'I Believe In You', 'artist': 'Robert...","{'song': 'Midnighter', 'artist': 'The Champs',...","{'song': 'Chariot Rock', 'artist': 'The Champs...","{'song': 'Down In Virginia', 'artist': 'Jimmy ...","{'song': 'Sunday Barbecue', 'artist': 'Tenness...","{'song': 'Gotta Have Rain', 'artist': 'Eydie G...","{'song': 'Nothing In The World', 'artist': 'Na...","{'song': 'Baubles, Bangles And Beads', 'artist..."
2,"{'song': 'Nel Blu Dipinto Di Blu (Volare)', 'a...","{'song': 'Little Star', 'artist': 'The Elegant...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Just A Dream', 'artist': 'Jimmy Clan...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...","{'song': 'Rebel-'rouser', 'artist': 'Duane Edd...","{'song': 'Fever', 'artist': 'Peggy Lee', 'this...","{'song': 'Splish Splash', 'artist': 'Bobby Dar...",...,"{'song': 'Down The Aisle Of Love', 'artist': '...","{'song': 'Return To Me', 'artist': 'Dean Marti...","{'song': 'Down In Virginia', 'artist': 'Jimmy ...","{'song': 'Fire Of Love', 'artist': 'Jody Reyno...","{'song': 'Put A Ring On My Finger', 'artist': ...","{'song': 'It's All In The Game', 'artist': 'To...","{'song': 'Ma Ma Ma Marie', 'artist': 'The Gayl...","{'song': 'Where The Blue Of The Night', 'artis...","{'song': 'Who Are They To Say', 'artist': 'The...","{'song': 'Going To Chicago Blues', 'artist': '..."
3,"{'song': 'Little Star', 'artist': 'The Elegant...","{'song': 'Nel Blu Dipinto Di Blu (Volare)', 'a...","{'song': 'Bird Dog', 'artist': 'The Everly Bro...","{'song': 'Just A Dream', 'artist': 'Jimmy Clan...","{'song': 'My True Love', 'artist': 'Jack Scott...","{'song': 'Poor Little Fool', 'artist': 'Ricky ...","{'song': 'Patricia', 'artist': 'Perez Prado An...","{'song': 'Born Too Late', 'artist': 'Poni-Tail...","{'song': 'Fever', 'artist': 'Peggy Lee', 'this...","{'song': 'When', 'artist': 'Kalin Twins', 'thi...",...,"{'song': 'Ramrod', 'artist': 'Duane Eddy His T...","{'song': 'Just Like In The Movies', 'artist': ...","{'song': 'The Purple People Eater', 'artist': ...","{'song': 'You Cheated', 'artist': 'The Shields...","{'song': 'No One Knows', 'artist': 'Dion & The...","{'song': 'Return To Me', '

In [7]:
new_dataset = pd.json_normalize(pd.melt(pd.json_normalize(file_new.data).transpose()).value)

In [8]:
new_dataset['date'] = new_dataset['date'][0]
k = 0
for i in file_new['date']:
    for j in range(0,100):
        new_dataset['date'][k] = i
        k = k+1


KeyError: 'date'

In [ ]:
new_dataset['date'] = 0
new_dataset['date'][0] = 1

In [ ]:
new_dataset['date'] = pd.to_datetime(new_dataset['date']).dt.date

In [ ]:
new_dataset.head(5)

In [ ]:
new_dataset = new_dataset.iloc[:, [6,0,1,2,3,4,5]]
new_dataset

In [ ]:

table = pa.Table.from_pandas(new_dataset)
pq.write_table(table, 'raw/billboard/billboard_100_OAT.parquet')

In [ ]:
new_dataset.to_json('raw/billboard/billboard_100_OAT.json')

In [ ]:
new_dataset.to_csv('raw/billboard/billboard_100_OAT.csv', index=False)